 Notebook: erp_ingest_invoices

 Layer: Ingestion → Bronze

 Source: ERP (CSV files from Unity Catalog Volume)

 Description:
 - Validates raw data (batch - commented)
 - Ingests data using Auto Loader (incremental)
 - Writes to Bronze Delta table

 Architecture:
 Volumes → Auto Loader → Bronze





In [0]:
source_path="/Volumes/workspace/default/raw_data/erp/invoices/"


Read raw CSV files from volume
- Reads all files in folder
- Uses correct delimiter (;)
- No transformations (raw ingestion)
--------------------------------------------

In [0]:
df_invoices=spark.read\
  .format("csv")\
  .option("header", "true")\
  .option("inferSchema", "true")\
  .option("delimiter", ";")\
  .load(source_path)


  Validate data


In [0]:
df_invoices.printSchema()
display(df_invoices)



  Row count check


In [0]:
df_invoices.count()


 Step 3: Define source and checkpoint paths


In [0]:

source_path="/Volumes/workspace/default/raw_data/erp/invoices/"
checkpoint_path="/Volumes/workspace/default/raw_data/checkpoints/invoices_checkpoint"
schema_location = "/Volumes/workspace/default/raw_data/schemas/invoices"


 Step 4: Define Schema


In [0]:
import pyspark.sql.types as T

invoices_schema=T.StructType([
    T.StructField("invoice_id",T.StringType(),True),
    T.StructField("order_id",T.StringType(),True),
    T.StructField("invoice_date",T.StringType(),True),
    T.StructField("amount_paid",T.IntegerType(),True),
    T.StructField("payment_method",T.StringType(),True),
    T.StructField("payment_status",T.StringType(),True)
])



 Step 5: Read invoices using Auto Loader (with metadata)
 - Incremental ingestion
 - Detects new files automatically
 - Adds ingestion metadata (timestamp + source file)
 - Handles schema evolution


In [0]:
import pyspark.sql.functions as F

df_invoices_stream = (
    spark.readStream
    .format("cloudfiles")
    .option("cloudfiles.format", "csv")
    .option("cloudFiles.schemaLocation", schema_location)       
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")  
    .option("cloudFiles.schemaHints", "invoice_id STRING, order_id STRING, invoice_date STRING, amount_paid INT, payment_method STRING, payment_status STRING")  
    .option("header", True)
    .option("delimiter", ";")
    .load(source_path)
    .withColumn("ingestion_timestamp", F.current_timestamp())
    .withColumn("source_file", F.col("_metadata.file_path"))
)


 Step 6: Write stream to Bronze Delta table
 - Stores raw ERP invoices
 - Append-only (no transformations)
 - Checkpoint ensures incremental processing


In [0]:
(
    df_invoices_stream
    .writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .outputMode("append")
    .trigger(availableNow=True)
    .toTable("bronze.erp_invoices")
)


 Step 7: Validate Bronze table
 - Verify ingestion worked
 - Check metadata columns


In [0]:
display(spark.table("bronze.erp_invoices"))

In [0]:
%sql
Select count(*)
from bronze.erp_invoices